In [94]:
""" Lakshita's input code"""

with open('ASHA_New_Basic_Report.tsv', 'r') as file:
    cancer_type = None
    sample_id = None
    biomarkers_data = []
    hrr_data = []
    read_biomarkers = False
    read_hrr = False
    spacing_encountered_biomarkers = False

    dna_sequence_variants_data = []
    copy_number_variations_data = []
    reading_dna_sequence_variants = False
    reading_copy_number_variations = False
    signal_line_space_count = 0

    for line in file:
        line = line.strip()
        if line.startswith("Sample Cancer Type"):
            cancer_type = line.split('\t')[1]

        elif line.lower().startswith("**sample id:**|"):
            sample_id = line.split('|')[1].strip()

        elif line.startswith("Tumor Mutational Burden"):
            mutational_burden = line.split('\t')[1]

        elif line == "Relevant Biomarkers":
            read_biomarkers = True
            continue
        elif read_biomarkers and not spacing_encountered_biomarkers:
            spacing_encountered_biomarkers = True
            continue
        elif read_biomarkers and line:
            biomarkers_data.append(line.split('\t'))
        elif read_biomarkers and spacing_encountered_biomarkers and not line:
            read_biomarkers = False
        elif line == "HRR Details":
            read_hrr = True
            continue
        elif read_hrr and ("Homologous recombination repair (HRR) genes were defined" in line or "Disclaimer" in line):
            read_hrr = False
            continue
        elif read_hrr and line:
            hrr_data.append(line.split('\t'))
        elif line == "DNA Sequence Variants":
            reading_dna_sequence_variants = True
            reading_copy_number_variations = False
            signal_line_space_count = 0
            continue
        elif line == "Copy Number Variations":
            reading_copy_number_variations = True
            reading_dna_sequence_variants = False
            signal_line_space_count = 0
            continue
        elif not line:
            signal_line_space_count += 1
            if signal_line_space_count == 2:
                reading_copy_number_variations = False
                continue
        elif reading_dna_sequence_variants and line:
            dna_sequence_variants_data.append(line.split('\t'))
        elif reading_copy_number_variations and line:
            copy_number_variations_data.append(line.split('\t'))

biomarkers_columns = ["Genomic Alteration","Tier", "Allele Frequency", "Mut/WT Ratio",
           "Locus", "Transcript","Relevant Therapies In this cancer type",
           "Relevant Therapies In other cancer type", "Clinical Trials",
           "Prognostic significance", "Diagnostic significance"]
biomarkers_df = pd.DataFrame(columns=biomarkers_columns, data=biomarkers_data[1:])

hrr_columns = ["Gene/Genomic Alteration", "Finding"]
hrr_df = pd.DataFrame(columns=hrr_columns, data=hrr_data[1:])

dna_sequence_variants_df = pd.DataFrame(dna_sequence_variants_data[1:], columns=dna_sequence_variants_data[0])
dna_sequence_variants_df=dna_sequence_variants_df.drop(columns=['Tier', 'Transcript', 'Allele Frequency', 'Variant ID'])
dna_sequence_variants_merged_df = pd.merge(biomarkers_df, dna_sequence_variants_df, on="Locus")
copy_number_variations_df = pd.DataFrame(copy_number_variations_data[1:], columns=copy_number_variations_data[0])
copy_number_variations_df=copy_number_variations_df.drop(columns=['Tier'])
copy_number_variations_merged_df = pd.merge(biomarkers_df, copy_number_variations_df, on="Locus")
# with open('/content/drive/MyDrive/Iris Thomas/ASHA_New_Basic_Report.tsv', 'r') as file:
#     lines = [line.strip() for line in file if not line.startswith('##')]
# filtered_in_df = pd.DataFrame([line.split('\t') for line in lines])
# filtered_in_df.columns = filtered_in_df.iloc[0]
# filtered_in_df = filtered_in_df[1:]
# selected_columns = ['Locus', 'Length', 'Oncomine Gene Class', 'Variant ID', 'CytoBand', 'ExAC GAF', 'dbSNP', 'SIFT', 'PolyPhen']
# filtered_in_df = filtered_in_df[selected_columns]
# with open('/content/drive/MyDrive/Iris Thomas/ASHA_New_Basic_Report.tsv', 'r') as file:
#     lines = [line.strip() for line in file if not line.startswith('##')]
# filtered_out_df = pd.DataFrame([line.split('\t') for line in lines])
# filtered_out_df.columns = filtered_out_df.iloc[0]
# filtered_out_df = filtered_out_df[1:]
# filtered_out_df = filtered_out_df[selected_columns]
# dna_sequence_variants_merged_with_filter_in_df = pd.merge(dna_sequence_variants_merged_df, filtered_in_df, on='Locus')
# dna_sequence_variants_merged_with_filter_out_df = pd.merge(dna_sequence_variants_merged_df, filtered_out_df, on='Locus')
# dna_sequence_variants_merged_with_filter_df = pd.concat([dna_sequence_variants_merged_with_filter_in_df, dna_sequence_variants_merged_with_filter_out_df])
# dna_sequence_variants_merged_with_filter_df = dna_sequence_variants_merged_with_filter_df.drop_duplicates(subset=['Locus'])
# dna_sequence_variants_merged_with_filter_df = dna_sequence_variants_merged_with_filter_df[~(dna_sequence_variants_merged_with_filter_df['Filter'].isin(['NOCALL', 'FAIL', 'NO CALL']))]

# copy_number_variations_merged_with_filter_in_df = pd.merge(copy_number_variations_merged_df, filtered_in_df, on='Locus')
# copy_number_variations_merged_with_filter_out_df = pd.merge(copy_number_variations_merged_df, filtered_out_df, on='Locus')
# copy_number_variations_merged_with_filter_df = pd.concat([copy_number_variations_merged_with_filter_in_df, copy_number_variations_merged_with_filter_out_df])
# copy_number_variations_merged_with_filter_df = copy_number_variations_merged_with_filter_df.drop_duplicates(subset=['Locus'])
# copy_number_variations_merged_with_filter_df = copy_number_variations_merged_with_filter_df[~(copy_number_variations_merged_with_filter_df['Filter'].isin(['NOCALL', 'FAIL', 'NO CALL']))]

#print(dna_sequence_variants_df)
df1=pd.DataFrame(dna_sequence_variants_df)
print(df1)


     Gene Amino Acid Change       Coding           Locus Variant Effect  \
0  PIK3CA         p.(E545K)    c.1633G>A  chr3:178936091       missense   
1   FBXW7         p.(R465H)    c.1394G>A  chr4:153249384       missense   
2    TP53         p.(G244D)     c.731G>A   chr17:7577550       missense   
3   BRIP1             p.(?)  c.1474-3T>C  chr17:59861788        unknown   
4  CYP2D6             p.(?)   c.506-1G>A  chr22:42524947        unknown   

       Location                                       ClinVar Exon Coverage  \
0        exonic                  Pathogenic/Likely pathogenic   10     1998   
1        exonic                             Likely pathogenic    9     1986   
2        exonic                  Pathogenic/Likely pathogenic    7     1995   
3  splicesite_3  Conflicting interpretations of pathogenicity   11      492   
4  splicesite_3           Likely benign| drug response| other    4      984   

  Oncomine Variant Class Filter  
0                Hotspot   PASS  
1     

In [95]:
import pandas as pd
""" extract chromosome number and position and store in columns "Chr_no_mut_tast_input" and "Chr_pos_mut_taster_input" """
split_data = df1["Locus"].str[3:].str.split(":", expand=True)  # Split using ":" as delimiter
df1["Chr_no_mut_taster_input"] = split_data[0] if len(split_data.columns) >= 1 else 'NA'  # Handle missing values
df1["Chr_pos_mut_taster_input"] = split_data[1] if len(split_data.columns) >= 2 else 'NA'  # Handle missing values
print(df1)
print(df1.columns)



     Gene Amino Acid Change       Coding           Locus Variant Effect  \
0  PIK3CA         p.(E545K)    c.1633G>A  chr3:178936091       missense   
1   FBXW7         p.(R465H)    c.1394G>A  chr4:153249384       missense   
2    TP53         p.(G244D)     c.731G>A   chr17:7577550       missense   
3   BRIP1             p.(?)  c.1474-3T>C  chr17:59861788        unknown   
4  CYP2D6             p.(?)   c.506-1G>A  chr22:42524947        unknown   

       Location                                       ClinVar Exon Coverage  \
0        exonic                  Pathogenic/Likely pathogenic   10     1998   
1        exonic                             Likely pathogenic    9     1986   
2        exonic                  Pathogenic/Likely pathogenic    7     1995   
3  splicesite_3  Conflicting interpretations of pathogenicity   11      492   
4  splicesite_3           Likely benign| drug response| other    4      984   

  Oncomine Variant Class Filter Chr_no_mut_tast_input Chr_pos_mut_taster_i

In [96]:
import pandas as pd

def extract_string_after_last_number(text):
  """
  Extracts the string immediately following the last number in the given text.
  Constructs mutation taster query and store in "Mutation_taster_query" column

  Args:
      text: The text to extract the string from (can be a string or NaN).

  Returns:
      The string immediately following the last number, or None if no number is found or the input is NaN.

      
  """

  if pd.isna(text):  # Check for missing values (NaN)
    return None

  # Find the indices of all digits in the text
  digit_indices = [i for i, char in enumerate(text) if char.isdigit()]

  # If no digits are found, return None
  if not digit_indices:
    return None

  # Get the last digit index
  last_digit_index = digit_indices[-1]

  # Extract the string starting from the character after the last digit
  return text[last_digit_index + 1:].strip()

# Assuming your DataFrame is named "df"
ref_alt = []
ref=[]
alt=[]
for code in df["Coding"]:
  extracted_string = extract_string_after_last_number(code)
  ref_alt.append(extracted_string)
  print(extracted_string)  # Optional: Print extracted strings

# Print the list of extracted strings (if desired)
print(ref_alt)
for base in ref_alt:
    bases=base.split(">")
    ref.append(bases[0])
    alt.append(bases[1])
print (ref)
print (alt)
df1 ["Ref"]=ref
df1["Alt"]=alt
#print (df)


df1["Mutation_taster_query"] = "chromosome="+df1["Chr_no_mut_tast_input"]+"&position="+df1["Chr_pos_mut_taster_input"]+"&ref="+df1["Ref"]+"&alt="+df["Alt"]
print(df1)

G>A
G>A
G>A
T>C
G>A
['G>A', 'G>A', 'G>A', 'T>C', 'G>A']
['G', 'G', 'G', 'T', 'G']
['A', 'A', 'A', 'C', 'A']
     Gene Amino Acid Change       Coding           Locus Variant Effect  \
0  PIK3CA         p.(E545K)    c.1633G>A  chr3:178936091       missense   
1   FBXW7         p.(R465H)    c.1394G>A  chr4:153249384       missense   
2    TP53         p.(G244D)     c.731G>A   chr17:7577550       missense   
3   BRIP1             p.(?)  c.1474-3T>C  chr17:59861788        unknown   
4  CYP2D6             p.(?)   c.506-1G>A  chr22:42524947        unknown   

       Location                                       ClinVar Exon Coverage  \
0        exonic                  Pathogenic/Likely pathogenic   10     1998   
1        exonic                             Likely pathogenic    9     1986   
2        exonic                  Pathogenic/Likely pathogenic    7     1995   
3  splicesite_3  Conflicting interpretations of pathogenicity   11      492   
4  splicesite_3           Likely benign| drug 

In [97]:
import requests
from bs4 import BeautifulSoup
import urllib.parse  # Not used in this specific code, but potentially useful for future modifications

""" Mutation taster UPI
    loops through the query list """

# Define base URL
base_url = "https://www.genecascade.org/MTc2021/ChrPos102.cgi"
query_list = list(df1["Mutation_taster_query"])
print(query_list)
all_predictions_list=[]

for query in query_list:
    # Construct the complete URL
    url = f"{base_url}?{query}"  # Use URL parameters instead of query string concatenation

    response = requests.get(url)
    prediction_text_list=[]

    # Check for successful response
    if response.status_code == 200:
        # Parse HTML content with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the summary table
        summary_table = soup.find('table', class_='resultlinks')  # Assuming specific class for the table

        if summary_table is not None:
            # Initialize empty dictionary to store transcript-prediction pairs
            all_predictions = {}

            # Skip the header row (index 0)
            for row in summary_table.find_all('tr')[1:]:
                # Extract transcript ID from the first cell (assuming anchor tag)
                transcript_id_element = row.find('td')
                if transcript_id_element is not None:
                    transcript_id = transcript_id_element.text.strip()

                # Extract prediction from the third cell (assuming span tag)
                prediction_element = row.find_all('td')[2]
                if prediction_element is not None:
                    # Extract text from the prediction element's child (assuming it's a span)
                    prediction_text = prediction_element.find('span').text.strip()

                    # Add transcript-prediction pair to the dictionary
                    all_predictions[transcript_id] = prediction_text
                    #print(prediction_text)



            # Print extracted predictions
            if all_predictions:
                print("Transcript ID - Prediction:")
                for transcript_id, prediction in all_predictions.items():
                    print(f"{transcript_id}: {prediction}")
                    prediction_text_list.append(prediction)

            else:
                print("No predictions found in the table.")
        else:
            print("Summary table not found in the response.")

        #print (prediction_text_list)
        all_predictions_list.append(prediction_text_list)
        print(all_predictions_list)
        #prediction_text_list.clear()
        #print (prediction_text_list)
    else:
        print(f"Error: Failed to fetch URL: {url} (Status code: {response.status_code})")
dna_sequence_variants_df["Mutation_taster_results"] = [", ".join(p) for p in all_predictions_list]

#dna_sequence_variants_df["Mutation_taster_results"]= all_predictions_list
print(dna_sequence_variants_df)
dna_sequence_variants_df.to_excel("mutation_taster.xlsx")


['chromosome=3&position=178936091&ref=G&alt=A', 'chromosome=4&position=153249384&ref=G&alt=A', 'chromosome=17&position=7577550&ref=G&alt=A', 'chromosome=17&position=59861788&ref=T&alt=C', 'chromosome=22&position=42524947&ref=G&alt=A']
Transcript ID - Prediction:
ENST00000263967: Deleterious (ClinVar)
[['Deleterious (ClinVar)']]
Transcript ID - Prediction:
ENST00000296555: Deleterious (ClinVar)
ENST00000603841: Deleterious (ClinVar)
ENST00000603548: Deleterious (ClinVar)
ENST00000263981: Deleterious (ClinVar)
ENST00000281708: Deleterious (ClinVar)
ENST00000393956: Deleterious (ClinVar)
[['Deleterious (ClinVar)'], ['Deleterious (ClinVar)', 'Deleterious (ClinVar)', 'Deleterious (ClinVar)', 'Deleterious (ClinVar)', 'Deleterious (ClinVar)', 'Deleterious (ClinVar)']]
Transcript ID - Prediction:
ENST00000413465: Deleterious (ClinVar)
ENST00000359597: Deleterious (ClinVar)
ENST00000269305: Deleterious (ClinVar)
ENST00000455263: Deleterious (ClinVar)
ENST00000420246: Deleterious (ClinVar)
ENST0